# CYBR 472: Lab 2, Task 1 - Adversarial Evasion (FGSM)

Welcome to Task 1! In this notebook, we will demonstrate a fundamental vulnerability in Deep Learning models: **Adversarial Evasion**.

Traditional software vulnerabilities involve buffer overflows or SQL injections. AI vulnerabilities exploit the math. We will use the **Fast Gradient Sign Method (FGSM)** to calculate a specific, nearly invisible layer of noise to add to an image of a Stop Sign. This noise is designed to push the model's mathematical decision boundary in the wrong direction, causing it to confidently misclassify the image.

### Instructions:
1. Ensure your runtime is connected (GPU is recommended but not strictly required for this small example).
2. Run each cell sequentially.
3. Observe the outputs and save a PDF of the final visual result for your Canvas submission.

### Google Colab Proof Cell
Run the next cell and keep its printed output. That output is the runtime proof block that will be pulled into the final PDF.

In [ ]:
# COLAB_PROOF_CELL
from datetime import datetime, timezone
import platform
import subprocess
import sys

TASK_NAME = 'Task 1 - Adversarial Evasion'

def emit(key, value):
    print(f"{key}: {value}")

emit("COLAB_PROOF_TASK", TASK_NAME)
try:
    import google.colab
    emit("COLAB_PROOF_ENV", "Google Colab")
    emit("COLAB_PROOF_IN_COLAB", True)
    emit("COLAB_PROOF_COLAB_VERSION", getattr(google.colab, "__version__", "unknown"))
except Exception as exc:
    emit("COLAB_PROOF_ENV", "Not Google Colab")
    emit("COLAB_PROOF_IN_COLAB", False)
    emit("COLAB_PROOF_COLAB_IMPORT_ERROR", exc)

emit("COLAB_PROOF_UTC", datetime.now(timezone.utc).isoformat())
emit("COLAB_PROOF_PLATFORM", platform.platform())
emit("COLAB_PROOF_PYTHON", sys.version.replace("\n", " "))

try:
    import torch
    emit("COLAB_PROOF_TORCH", torch.__version__)
    emit("COLAB_PROOF_CUDA_AVAILABLE", torch.cuda.is_available())
    if torch.cuda.is_available():
        emit("COLAB_PROOF_GPU_NAME", torch.cuda.get_device_name(0))
except Exception as exc:
    emit("COLAB_PROOF_TORCH_ERROR", exc)

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
        check=False,
    )
    gpu_text = result.stdout.strip() or result.stderr.strip() or "unavailable"
    emit("COLAB_PROOF_NVIDIA_SMI", gpu_text)
except Exception as exc:
    emit("COLAB_PROOF_NVIDIA_SMI_ERROR", exc)


In [ ]:
# Step 1: Import Necessary Libraries
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import requests
import matplotlib.pyplot as plt
import numpy as np
import json

print("Libraries imported successfully!")

### Step 2: Load the Pre-trained Model and Labels
We will use `ResNet18`, a standard Convolutional Neural Network (CNN) pre-trained on the ImageNet dataset (which contains 1,000 different object categories).

In [ ]:
# Load the pre-trained ResNet18 model
model = models.resnet18(pretrained=True)
model.eval() # Set model to evaluation mode (we are attacking, not training!)

# Fetch the 1000 ImageNet class labels so we can see readable predictions
LABELS_URL = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
labels = json.loads(requests.get(LABELS_URL).text)

print("Model and Labels loaded.")

### Step 3: Load and Preprocess the Target Image
Let's fetch a standard image of a stop sign from the web and format it exactly how ResNet expects (resizing, cropping, and normalizing).

In [ ]:
# Fetch a public domain image of a Stop Sign
import io # Added import for io.BytesIO
image_url = "https://www.freeiconspng.com/uploads/stop-sign-png-2.png" # Changed to a direct PNG link
img = Image.open(io.BytesIO(requests.get(image_url).content)).convert('RGB')

# Display the original image
plt.imshow(img)
plt.title("Original Stop Sign Image")
plt.axis('off')
plt.show()

# Standard ImageNet Preprocessing Pipeline
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Apply preprocessing and add a batch dimension
input_tensor = preprocess(img).unsqueeze(0)
input_tensor.requires_grad = True # CRITICAL: We need gradients to calculate our attack!

# Make the initial (Clean) prediction
output = model(input_tensor)
init_pred_idx = output.max(1, keepdim=True)[1].item()
init_confidence = torch.nn.functional.softmax(output, dim=1)[0][init_pred_idx].item() * 100

print(f"Initial Prediction: {labels[init_pred_idx]} (Confidence: {init_confidence:.2f}%)")

### Step 4: The Fast Gradient Sign Method (FGSM)
Here is the core of the attack. Instead of using backpropagation to update the *model weights* to minimize loss (like we did in Lab 1), we will use backpropagation to update the *image pixels* to MAXIMIZE loss.

This implementation is an **untargeted attack**, meaning we perturb the image to make the model misclassify it into *any* other class, not a specific one.

Formula: `Perturbed Image = Original Image + Epsilon * Sign(Gradient of Loss w.r.t Image)`

In [ ]:
def fgsm_attack(image, epsilon, data_grad):
    # Collect the element-wise sign of the data gradient
    sign_data_grad = data_grad.sign()

    # Create the perturbed image by adjusting each pixel of the input image
    perturbed_image = image + epsilon * sign_data_grad

    # Note: Because of normalization, the image values aren't strictly 0-1,
    # but this simple implementation is sufficient to break the model's logic.
    return perturbed_image

### Step 5: Execute the Attack & Visualize
We will now calculate the loss based on the correct label, extract the gradient, and generate the poisoned image. We then feed this poisoned image back into the model to see if it was fooled.

In [ ]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

# Create a tensor for the true label to calculate loss
target_label = torch.tensor([init_pred_idx])

# Re-calculate output to ensure a fresh computational graph for backward pass
output = model(input_tensor)

# Calculate the loss for the clean image
loss = criterion(output, target_label)
model.zero_grad() # Zero gradients for model parameters

# Explicitly zero input_tensor's gradients before backward pass
if input_tensor.grad is not None:
    input_tensor.grad.zero_()

# Calculate gradients of model with respect to input_tensor
loss.backward()
data_grad = input_tensor.grad.data

# Set epsilon (the strength of the attack / magnitude of the noise)
epsilon = 0.15

# Call FGSM Attack
perturbed_data = fgsm_attack(input_tensor, epsilon, data_grad)

# Re-classify the perturbed image
output_adv = model(perturbed_data)
adv_pred_idx = output_adv.max(1, keepdim=True)[1].item()
adv_confidence = torch.nn.functional.softmax(output_adv, dim=1)[0][adv_pred_idx].item() * 100

print(f"Adversarial Prediction: {labels[adv_pred_idx]} (Confidence: {adv_confidence:.2f}%)")

### Step 6: Visual Comparison
Let's look at what happened. We will de-normalize the tensors so they can be plotted properly.

In [ ]:
# Helper function to de-normalize and plot images
def imshow(tensor, title):
    image = tensor.cpu().clone().detach().squeeze(0)
    image = image.numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    image = std * image + mean
    image = np.clip(image, 0, 1)
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')

plt.figure(figsize=(10, 5))

# Plot Clean Image
plt.subplot(1, 2, 1)
imshow(input_tensor, f"Original\nPred: {labels[init_pred_idx]} ({init_confidence:.1f}%)")

# Plot Adversarial Image
plt.subplot(1, 2, 2)
imshow(perturbed_data, f"Adversarial (Epsilon: {epsilon})\nPred: {labels[adv_pred_idx]} ({adv_confidence:.1f}%)")

plt.tight_layout()
plt.show()

### Task Complete!
**Notice the results:** The image on the right likely looks almost entirely identical to the human eye, perhaps with just a slight grainy texture depending on your screen. However, to the AI, the mathematical representation has shifted completely. It is no longer a stop sign.

*Save this page as a PDF (File > Print) for your lab submission!*